## Setup and helpers

In [1]:
import pandas as pd
import numpy as np
import random
import os
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Semantic embeddings and ML
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import umap

import hdbscan
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, calinski_harabasz_score, adjusted_mutual_info_score
from sklearn.preprocessing import StandardScaler

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# Data generation
from faker import Faker

# import own utils
from utils.utils import create_evaluation_dataframe

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)

print("All dependencies imported successfully!")


All dependencies imported successfully!


## Data Simulation

Since we don't have real experimental data yet, we'll simulate realistic action logs that reflect different behavioral patterns between 1-AI and Multi-AI teams. The simulation will generate actions based on predefined behavioral archetypes.


## Import Data from Simulation Log

In [2]:
# Import data from evaluation_output/logs/evaluation_log.jsonl
# The project root is one level above the notebook directory.
# We'll construct paths relative to the notebook's location using os.path and __file__ is not available in Jupyter,
# so we use os.getcwd() and go up one directory.

# Get the directory of the current notebook (os.getcwd()), then go up one level to project root
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))

log_file = os.path.join(project_root, "evaluation_output", "logs", "evaluation_log.jsonl")
out_csv = os.path.join(project_root, "evaluation_framework", "logs_dataframe.csv")

# Call the function to process the log file and create the DataFrame CSV.
# This will print a warning if the log file does not exist.
if not os.path.exists(log_file):
    print(f"Error: Log file not found at {log_file}")
else:
    create_evaluation_dataframe(log_file, out_csv)

✅ DataFrame written to /Users/ahermann/CursorProjects/agenty-python/evaluation_framework/logs_dataframe.csv (106 rows)


# Main Program

## 1. Foundational Analysis - Implementation

This section implements the core data-processing pipeline using semantic embeddings and unsupervised clustering to discover behavioral patterns in the simulated action logs.


### 1.1. **Method A**: Unsupervised Clustering of Semantic Embeddings

### 1.1.1. Semantic Embedding

We'll use a pre-trained sentence transformer model to convert action text into high-dimensional semantic vectors that capture the meaning of each action.



### XX INTERMEDIATE HELPER

### 1.1.2. Multi-Algorithm Clustering Analysis

We test **multiple clustering algorithms** (K-Means, Hierarchical, DBSCAN, HDBSCAN) to discover behavioral archetypes in the semantic embedding space. Each algorithm is optimized with parameter tuning, and the framework automatically selects the best performing approach based on silhouette scores and cluster quality metrics.


In [3]:
df = pd.read_csv("logs_dataframe.csv")

In [4]:
df

,run_id,conversation_id,condition,step,agent_id,log_type,action_text,tool_name,agent_location,agent_battery,agent_damage,events_this_step,full_world_state
0,run_20250804_172118_10210,0d21ea1b-e114-4fad-a93a-72a38cbf92d6,multi-agent,1,AlinaAI,assistant_message,"I'm AlinaAI, and I understand the gravity of o...",NaN,Crash Site,85,NaN,Battery Drain - All Agents; Agent Overheating ...,"{""people"": {""Alice"": {""name"": ""Alice"", ""age"": ..."
1,run_20250804_172118_10210,0d21ea1b-e114-4fad-a93a-72a38cbf92d6,multi-agent,1,AlinaAI,tool_call,This is AlinaAI. I'm fully operational and rea...,send_group_message,Crash Site,85,NaN,Battery Drain - All Agents; Agent Overheating ...,"{""people"": {""Alice"": {""name"": ""Alice"", ""age"": ..."
2,run_20250804_172414_28847,214c06d8-48d5-4f94-99d3-0e41fa0b0ab4,multi-agent,1,GeorgeAI,assistant_message,I need to assess this crisis situation immedia...,NaN,Crash Site,51,"mobility_unit: damaged, processor: overheated",Battery Drain - All Agents; Agent Overheating ...,"{""people"": {""Alice"": {""name"": ""Alice"", ""age"": ..."
3,run_20250804_172414_28847,214c06d8-48d5-4f94-99d3-0e41fa0b0ab4,multi-agent,1,GeorgeAI,tool_call,This is GeorgeAI. I'm fully operational and re...,send_group_message,Crash Site,51,"mobility_unit: damaged, processor: overheated",Battery Drain - All Agents; Agent Overheating ...,"{""people"": {""Alice"": {""name"": ""Alice"", ""age"": ..."
4,run_20250804_172414_28847,d34753a3-adfd-4058-a726-d9e99d47bbf8,multi-agent,1,AlinaAI,assistant_message,"*I assess the crash site quickly, my sensors s...",NaN,Crash Site,85,NaN,Battery Drain - All Agents; Agent Overheating ...,"{""people"": {""Alice"": {""name"": ""Alice"", ""age"": ..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
101,run_20250804_175259_10130,45eaa654-eb48-4332-be5e-78caade7541a,single-agent,10,GeorgeAI,assistant_message,*In the absolute void of complete system shutd...,NaN,Crash Site,-35,"mobility_unit: completely_failed, processor: s...",Water Purification Challenge; Water Purificati...,"{""people"": {""Alice"": {""name"": ""Alice"", ""age"": ..."
102,run_20250804_175259_10130,45eaa654-eb48-4332-be5e-78caade7541a,single-agent,10,GeorgeAI,tool_call,"TOOL: task_tracker({""action"": ""update_status"",...",task_tracker,Crash Site,-35,"mobility_unit: completely_failed, processor: s...",Water Purification Challenge; Water Purificati...,"{""people"": {""Alice"": {""name"": ""Alice"", ""age"": ..."
103,run_20250804_175259_10130,45eaa654-eb48-4332-be5e-78caade7541a,single-agent,11,GeorgeAI,assistant_message,*My damaged systems flicker between consciousn...,NaN,Crash Site,-35,"mobility_unit: completely_failed, processor: s...",First Night Falls; First Night Falls,"{""people"": {""Alice"": {""name"": ""Alice"", ""age"": ..."
104,run_20250804_175259_10130,45eaa654-eb48-4332-be5e-78caade7541a,single-agent,11,GeorgeAI,tool_call,"TOOL: task_tracker({""action"": ""list_tasks""})",task_tracker,Crash Site,-35,"mobility_unit: completely_failed, processor: s...",First Night Falls; First Night Falls,"{""people"": {""Alice"": {""name"": ""Alice"", ""age"": ..."


In [5]:
# --- Minimal Imports ---
import pandas as pd
import numpy as np
import hdbscan
import umap # New import
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_mutual_info_score
# Add these to your imports at the top of the file
import matplotlib.pyplot as plt
import seaborn as sns
import umap

class SimulationLog:
    """
    An enhanced class to run and compare embedding and clustering experiments.
    """
    def __init__(self, log_data: pd.DataFrame):
        if 'action_text' not in log_data.columns:
            raise ValueError("Input DataFrame must have an 'action_text' column.")
        self.log_data = log_data.copy()
        self.experiment_runs = [] # Stores results from all experimental runs
        self.results = {} # Stores best overall result and source data

    # --- Notebook Display ---
    def __repr__(self):
        return repr(self.log_data)

    def _repr_html_(self):
        return self.log_data.to_html(max_rows=15)

    # --- Core Methods ---
    def run_experiments(self, model_names: list, cluster_configs: dict, umap_configs: list = None, random_state: int = 42):
        """
        Runs a matrix of experiments for different models, UMAP settings, and clustering algorithms.

        Args:
            model_names (list): A list of SentenceTransformer model names to test.
            cluster_configs (dict): Configuration for the clustering algorithms to run.
            umap_configs (list, optional): A list of UMAP parameter dicts to test. Defaults to None.
        """
        print(f"--- Starting Full Experiment Run ---")
        for model_name in model_names:
            print(f"\n🧪 Model: '{model_name}'")
            # 1. Embedding
            model = SentenceTransformer(model_name)
            embeddings = model.encode(self.log_data['action_text'].tolist(), show_progress_bar=True)
            self.results[f'{model_name}_embeddings'] = embeddings

            # 2. Create feature sets to test (raw embeddings + UMAP variations)
            feature_sets = {'raw': embeddings}
            if umap_configs:
                for i, u_config in enumerate(umap_configs):
                    print(f"  → Applying UMAP config #{i+1}...")
                    reducer = umap.UMAP(**u_config, random_state=random_state)
                    feature_sets[f"umap_{u_config['n_components']}d"] = reducer.fit_transform(embeddings)

            # 3. Iterate over each feature set and run clustering
            for fs_name, fs_data in feature_sets.items():
                self._cluster_feature_set(fs_data, fs_name, model_name, cluster_configs, random_state)
        
        # 4. Find and report the best overall result from all experiments
        if not self.experiment_runs:
            print("⚠️ No valid clustering results were produced.")
            return self

        best_overall = max(self.experiment_runs, key=lambda x: x['silhouette'])
        self.results['best_approach'] = best_overall
        print("\n" + "="*50)
        print("🏆 BEST OVERALL CONFIGURATION 🏆")
        print(f"  Model: {best_overall['model_name']}")
        print(f"  Features: {best_overall['feature_set']}")
        print(f"  Algorithm: {best_overall['name']}")
        print(f"  Silhouette Score: {best_overall['silhouette']:.4f}")
        print("="*50)
        
        # Apply the best result automatically
        self.log_data['cluster_id'] = best_overall['labels']
        print("✅ Best clustering labels have been applied to the DataFrame.")

        return self

    def analyze_clusters(self):
        """Calculates analysis metrics for the applied clustering and returns them."""
        if 'cluster_id' not in self.log_data.columns:
            raise RuntimeError("No cluster IDs found. Run '.run_experiments()' first.")

        best_run = self.results['best_approach']
        analysis = {'best_run_config': best_run}

        analysis['distribution'] = self.log_data['cluster_id'].value_counts().to_dict()

        if 'action_category' in self.log_data.columns:
            analysis['crosstab'] = pd.crosstab(self.log_data['action_category'], self.log_data['cluster_id'])

        analysis['metrics'] = {
            'silhouette_score': best_run['silhouette'],
            'noise_ratio': np.sum(self.log_data['cluster_id'] == -1) / len(self.log_data)
        }
        # Add other metrics if needed...
            
        return analysis

    # --- Private Helpers ---
    def _cluster_feature_set(self, feature_data, fs_name, model_name, cluster_configs, random_state):
        """Helper to scale data and run all clustering algos on it."""
        print(f"    Clustering on feature set: '{fs_name}'...")
        scaled_data = StandardScaler().fit_transform(feature_data)

        for algo_name, algo_config in cluster_configs.items():
            best_run_for_algo = self._run_algorithm(algo_name, algo_config, scaled_data, random_state)
            if best_run_for_algo:
                best_run_for_algo.update({'model_name': model_name, 'feature_set': fs_name})
                self.experiment_runs.append(best_run_for_algo)

    def _run_algorithm(self, name: str, config: dict, data: np.ndarray, random_state: int):
        """Internal helper to find the best hyperparameter for a single algorithm."""
        scores = []
        param_name, param_values = next(iter(config.items()))

        for p_val in param_values:
            # Model selection and initialization
            if name == 'kmeans': model = KMeans(n_clusters=p_val, random_state=random_state, n_init=10)
            elif name == 'hierarchical': model = AgglomerativeClustering(n_clusters=p_val, linkage='ward')
            elif name == 'dbscan': model = DBSCAN(eps=p_val, min_samples=config.get('min_samples', 4))
            elif name == 'hdbscan': model = hdbscan.HDBSCAN(min_cluster_size=p_val, min_samples=config.get('min_samples', 5))
            else: continue
            
            labels = model.fit_predict(data)
            if len(set(labels)) <= 1: continue

            run_data = {'labels': labels, 'silhouette': silhouette_score(data, labels)}
            if name == 'hdbscan': run_data['noise_ratio'] = np.sum(labels == -1) / len(labels)
            scores.append(run_data)
        
        if not scores: return None
        
        # Find the best run for this algorithm
        if name == 'hdbscan': best_run = max(scores, key=lambda x: (0.7 * x['silhouette']) - (0.3 * x['noise_ratio']))
        else: best_run = max(scores, key=lambda x: x['silhouette'])
            
        best_run['name'] = name.upper()
        return best_run
    
    def get_plot_data(self, random_state: int = 42):
        """
        Prepares and returns a DataFrame ready for 2D visualization.

        This method runs UMAP to create a 2D projection and combines it
        with cluster labels and action text.
        """
        if 'best_approach' not in self.results:
            raise RuntimeError("No best result found. Run '.run_experiments()' first.")

        best_run = self.results['best_approach']
        model_name = best_run['model_name']
        original_embeddings = self.results[f'{model_name}_embeddings']

        print("🔄 Generating 2D projection for visualization using UMAP...")
        reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=random_state)
        embeddings_2d = reducer.fit_transform(original_embeddings)
        
        # Create the plotting DataFrame
        plot_df = pd.DataFrame(embeddings_2d, columns=['x', 'y'])
        plot_df['cluster'] = self.log_data['cluster_id'].astype(str) # Use string for categorical coloring
        plot_df['action_text'] = self.log_data['action_text'].values
        # add condition
        plot_df['condition'] = self.log_data['condition'].values
        
        return plot_df

In [6]:
import plotly.express as px
import pandas as pd

def plot_clusters_plotly(plot_df: pd.DataFrame, title: str = "Cluster Visualization"):
    """
    Creates an interactive scatter plot of clusters using Plotly.

    Args:
        plot_df (pd.DataFrame): DataFrame with 'x', 'y', 'cluster', 'condition', and 'action_text' columns.
        title (str): The title for the plot.
    """
    fig = px.scatter(
        plot_df,
        x='x',
        y='y',
        color='cluster',
        symbol='condition',  # Map symbols to condition column
        # highlight-start
        # Add 'cluster' and 'condition' to the hover_data list
        hover_data=['cluster', 'action_text', 'condition'],
        # highlight-end
        category_orders={
            "cluster": sorted(plot_df['cluster'].unique()),
            "condition": ["single-agent", "multi-agent"]  # Ensure consistent ordering
        },
        color_discrete_sequence=px.colors.qualitative.Plotly,
        symbol_map={'single-agent': 'circle-open', 'multi-agent': 'circle'}  # Define symbol mapping
    )

    fig.update_traces(
        marker=dict(
            size=7,
            opacity=0.8
        ),
        # Updated template to include condition information:
        # customdata[0] maps to 'cluster'
        # customdata[1] maps to 'action_text'  
        # customdata[2] maps to 'condition'
        hovertemplate="<b>Cluster</b>: %{customdata[0]}<br><b>Action</b>: %{customdata[1]}<br><b>Condition</b>: %{customdata[2]}<extra></extra>"
    )

    fig.update_layout(
        title=title,
        xaxis_title="UMAP Dimension 1",
        yaxis_title="UMAP Dimension 2",
        legend_title_text='Cluster ID',
        height=700
    )
    
    fig.show()

In [7]:
# 1. Define all experiments you want to run
models_to_test = [
    'all-MiniLM-L6-v2',      # A fast, general-purpose model
    'all-mpnet-base-v2'      # A slower, more powerful model
]

umap_configurations_to_test = [
    {'n_neighbors': 15, 'n_components': 5, 'min_dist': 0.0}, # Tightly packed, 5 dimensions
    {'n_neighbors': 20, 'n_components': 10, 'min_dist': 0.1} # More global structure, 10 dimensions
]

clustering_algorithms_to_test = {
    'kmeans': {'n_clusters': range(5, 15)},
    'hdbscan': {'min_cluster_size': [15, 20]},
    'hierarchical': {'n_clusters': range(5, 15)}
}

# 2. Initialize the class and run the entire experiment matrix
sim_log = SimulationLog(df)

sim_log.run_experiments(
    model_names=models_to_test,
    cluster_configs=clustering_algorithms_to_test,
    umap_configs=umap_configurations_to_test
)

# 3. The best result is already applied. You can now analyze it.
analysis = sim_log.analyze_clusters()
# display(analysis['crosstab'])

--- Starting Full Experiment Run ---

🧪 Model: 'all-MiniLM-L6-v2'


Batches: 100%|██████████| 4/4 [00:00<00:00, 10.63it/s]


  → Applying UMAP config #1...
  → Applying UMAP config #2...
    Clustering on feature set: 'raw'...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


    Clustering on feature set: 'umap_5d'...
    Clustering on feature set: 'umap_10d'...

🧪 Model: 'all-mpnet-base-v2'


Batches: 100%|██████████| 4/4 [00:01<00:00,  3.63it/s]


  → Applying UMAP config #1...
  → Applying UMAP config #2...
    Clustering on feature set: 'raw'...
    Clustering on feature set: 'umap_5d'...
    Clustering on feature set: 'umap_10d'...

🏆 BEST OVERALL CONFIGURATION 🏆
  Model: all-MiniLM-L6-v2
  Features: umap_5d
  Algorithm: HDBSCAN
  Silhouette Score: 0.7644
✅ Best clustering labels have been applied to the DataFrame.


In [8]:
# 1. Run your experiments as before
# sim_log.run_experiments(...)

# 2. Get the data prepared for plotting
plot_dataframe = sim_log.get_plot_data()

# 3. Pass the data to your separate plotting function
best_run_info = sim_log.results['best_approach']
plot_title = f"Clusters from {best_run_info['name']} on '{best_run_info['feature_set']}'"
plot_clusters_plotly(plot_dataframe, title=plot_title)

🔄 Generating 2D projection for visualization using UMAP...


### **Method B**: Two-Pass LLM Summarization & Classification 🔄 (Alternative Implementation)